In [1]:
# Biblioteca para operações numéricas
import numpy as np

# Para visualização
import matplotlib.pyplot as plt

# Para redes neurais
import tensorflow as tf

# Interface de alto nível para construir redes neurais
from tensorflow import keras

# Importando as camadas da rede
from tensorflow.keras.layers import Conv2D, Input, MaxPooling2D, \
 Flatten, Dense, Dropout, BatchNormalization, Activation, \
 GlobalAveragePooling2D, RandomFlip, RandomTranslation, RandomRotation

# Regularização L2 e callbacks de treino
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Funções utilitárias para trabalhar com imagens
from tensorflow.keras.utils import load_img, img_to_array

# Semente para resultados mais reprodutíveis
keras.utils.set_random_seed(42)

# Carregar a base de Dados CIFAR-10

In [ ]:
# Carregar a base CIFAR
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

# Normalizar os valores para um intervalo [0,1]
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# y vem como coluna.
y_train = y_train.flatten()
y_test = y_test.flatten()

class_names = ["avião", "automóvel", "pássaro", "gato", "cervo",
               "cachorro", "sapo", "cavalo", "navio", "caminhão"]

print(f"Formato de X_train: ", X_train.shape)
print(f"Formato de y_train: ", y_train.shape)
print(f"Formato de X_test: ", X_test.shape)
print(f"Formato de y_test: ", y_test.shape)

# Plotar Algumas Imagens

In [ ]:
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis('off')
plt.suptitle("Exemplos da base CIFAR-10.")
plt.tight_layout()
plt.show()

# Criação do Modelo CNN (versão melhorada)

O que mudou em relação à versão anterior (~65% de acurácia):

1. **Data augmentation**: flip horizontal, pequenas translações e rotações. A rede vê variações das imagens a cada época e decora menos o treino. Essas camadas só atuam no treino; na avaliação e na predição elas não fazem nada.
2. **Mais filtros e duas convoluções por bloco** (32 → 64 → 128), com `padding='same'` para não encolher a imagem a cada convolução. O modelo antigo terminava com uma imagem 2x2, perdendo muita informação.
3. **BatchNormalization** depois de cada convolução: deixa o treino mais estável e rápido.
4. **Dropout crescente** por bloco e **regularização L2** nos pesos, para combater overfitting.
5. **GlobalAveragePooling2D** no lugar do `Flatten`: menos parâmetros na parte densa e menos overfitting.

In [ ]:
# Camadas de data augmentation (ativas apenas durante o treino)
data_augmentation = keras.Sequential([
    RandomFlip("horizontal"),
    RandomTranslation(0.1, 0.1),
    RandomRotation(0.05),
], name="data_augmentation")

L2 = regularizers.l2(1e-4)

def bloco_conv(filtros, dropout):
    """Conv -> BN -> ReLU -> Conv -> BN -> ReLU -> MaxPool -> Dropout"""
    return [
        Conv2D(filtros, (3, 3), padding='same', use_bias=False, kernel_regularizer=L2),
        BatchNormalization(),
        Activation('relu'),
        Conv2D(filtros, (3, 3), padding='same', use_bias=False, kernel_regularizer=L2),
        BatchNormalization(),
        Activation('relu'),
        MaxPooling2D(pool_size=(2, 2)),
        Dropout(dropout),
    ]

model = keras.Sequential(
    [Input(shape=(32, 32, 3)), data_augmentation]
    + bloco_conv(32, 0.2)    # Bloco 1: 32x32 -> 16x16
    + bloco_conv(64, 0.3)    # Bloco 2: 16x16 -> 8x8
    + bloco_conv(128, 0.4)   # Bloco 3: 8x8 -> 4x4
    + [
        # Transição CNN => Dense (Totalmente conectada)
        GlobalAveragePooling2D(),
        Dense(128, use_bias=False, kernel_regularizer=L2),
        BatchNormalization(),
        Activation('relu'),
        Dropout(0.5),
        Dense(10, activation='softmax'),
    ],
    name="cnn_cifar10"
)

# Compilar
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Resumo da arquitetura do modelo
model.summary()

# Treinar a Rede

Com augmentation o modelo precisa de mais épocas para convergir, então o treino vai até 60 épocas, com três callbacks:

- **ReduceLROnPlateau**: reduz a taxa de aprendizado pela metade quando a `val_loss` para de melhorar.
- **EarlyStopping**: interrompe o treino se a `val_accuracy` não melhorar por 12 épocas e restaura os melhores pesos.
- **ModelCheckpoint**: salva o melhor modelo em `melhor_modelo_cifar10.keras`.

In [ ]:
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True, verbose=1),
    ModelCheckpoint('melhor_modelo_cifar10.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1),
]

history = model.fit(X_train, y_train, epochs=60, batch_size=64,
                    validation_split=0.1, callbacks=callbacks)

# Plot da Acurácia e Loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label="Treino")
plt.plot(history.history['val_accuracy'], label="Validação")
plt.title("Acurácia por época")
plt.xlabel("Épocas")
plt.ylabel("Acurácia")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label="Treino")
plt.plot(history.history['val_loss'], label="Validação")
plt.title("Loss por época")
plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.legend()
plt.show()

# REALIZANDO PREDIÇÕES

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Acurácia no conjunto de teste: {test_acc * 100:.2f}%")
print(f"Loss no conjunto de teste: {test_loss:.4f}")

In [ ]:
pred_prob = model.predict(X_test[:9])  # Fazendo previsões para as primeiras 9 imagens do conjunto de teste
pred_labels = np.argmax(pred_prob, axis=1)  # Obtendo os rótulos previstos a partir das probabilidades
plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(X_test[i])
    real = class_names[y_test[i]]
    pred = class_names[pred_labels[i]]
    plt.title(f"Real: {real}\nPredição: {pred}")
    plt.axis('off')
plt.suptitle("Predições do modelo para as primeiras 9 imagens do conjunto de teste.")
plt.tight_layout()
plt.show()

# Matriz de Confusão

Mostra em quais classes o modelo mais erra e com quais ele as confunde (ex.: gato × cachorro).

In [ ]:
# Predições em todo o conjunto de teste
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

# Matriz de confusão (linhas = classe real, colunas = classe prevista)
cm = tf.math.confusion_matrix(y_test, y_pred, num_classes=10).numpy()

# Normalizada por linha: % de cada classe real que foi prevista como cada classe
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe real")
ax.set_title("Matriz de confusão (conjunto de teste)")

# Escreve a contagem e o percentual em cada célula
for i in range(10):
    for j in range(10):
        cor = 'white' if cm_norm[i, j] > 0.5 else 'black'
        ax.text(j, i, f"{cm[i, j]}\n{cm_norm[i, j]:.0%}",
                ha='center', va='center', color=cor, fontsize=8)

plt.tight_layout()
plt.show()

# Acurácia por classe (diagonal da matriz normalizada)
print("Acurácia por classe:")
for nome, acc in sorted(zip(class_names, cm_norm.diagonal()), key=lambda x: x[1]):
    print(f"  {nome:<10} {acc * 100:.1f}%")